# 10年定着予測 - AutoGluon シード平均 + bag16（62_）

## 位置づけ: `88.`のEDA結論「伸びしろは特徴量ではなくモデリング側」に沿った一手

2026-08-15の徹底EDA（`submit_result_report.md` 第88節 / [[monthly-data-information-ceiling]]）で、
**月次データには10年定着の情報がほとんど無い**ことが機構的に確定した（アウトカムに3年近い窓を
使ってすら定数予測から0.009しか改善しない）。生存時間モデリング一族も完全に閉じた。
したがって残る伸びしろはモデリング・アンサンブル側にある。

## なぜシード平均か

[[refit-chaos-noise-floor]]で「特徴量行列を少しでも変えるとTest予測が約0.014 MAD動き、
これはシードを引き直したのと同等」と実測済み。提出ゲートに使っているノイズ床は**0.02122**で、
これは我々が追いかけている差（0.001〜0.004）より一桁大きい。つまり**現在のスコアの相当部分が
実行ごとのランダム性に支配されている**可能性がある。

独立実行をN回平均すれば予測の分散は約1/√Nに縮む。[[modeling-levers-beat-new-features]]では
単層CatBoostのシード平均は「保険であって期待値の向上ではない」（-0.0003）と結論したが、
**AutoGluon（8-fold bagging + stacking）でのシード平均は未実施**であり、単層CatBoostより
ずっと分散が大きい構造なので効果が違う可能性がある。

## 本ノートブックで測ること

1. **同一構成のAutoGluonを3シードで独立実行し、WeightedEnsemble予測を平均する**
   （`50_`＝現最良の生成元と同一構成。特徴量・除外モデル・stack設定は一切変えない）
2. **シード間の予測MADを実測する** —— AutoGluon自体のノイズ床が何桁なのかは未測定。
   もしシード間MADが0.02122に近ければ、「これまでの構成比較の大半はノイズを見ていた」ことになり、
   [[validation-asymmetry]]の非対称性の理由そのものを説明できる。これは負の結果でも価値が高い。
3. **`num_bag_folds=16`（`50_`の8から倍増）** —— foldを増やすとOOF推定が安定し、
   その上に乗るstacker/WeightedEnsembleの品質が上がる可能性がある。時間予算は据え置き。

## 想定実行時間

3シード × full run（7200秒/fit）+ bag16 × 1 = 概算 **8〜10時間**。
`fit_autogluon()`は`predictor.pkl`の存在チェックで再学習をスキップするので、
切断されても同じセルを再実行すれば完了済みの学習は再利用される。

## 判定方針

採否は**Publicのみ**（[[validation-asymmetry]]）。検証は足切り専用。
シード平均は「同一構成の分散削減」であって新しい仮説ではないため、
[[validation-asymmetry]]の「同じレシピをより完全に実行しただけの変更は的中率が高い」
パターンに該当する。


In [21]:
!pip install -q catboost optuna

In [22]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [23]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [25]:
SCRIPT_NAME = "62_autogluon_seed_averaging"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-15 15:29:11] [INFO] === [62_autogluon_seed_averaging] 実験開始 ===


INFO:62_autogluon_seed_averaging:=== [62_autogluon_seed_averaging] 実験開始 ===


[2026-08-15 15:29:11] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


INFO:62_autogluon_seed_averaging:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


[2026-08-15 15:29:11] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/62_autogluon_seed_averaging_checkpoint.csv


INFO:62_autogluon_seed_averaging:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/62_autogluon_seed_averaging_checkpoint.csv


[2026-08-15 15:29:11] [INFO] チェックポイントは未作成（新規実行）


INFO:62_autogluon_seed_averaging:チェックポイントは未作成（新規実行）


In [26]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


[2026-08-15 15:29:12] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:62_autogluon_seed_averaging:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-15 15:29:12] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:62_autogluon_seed_averaging:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-15 15:29:12] [INFO] 定着率: 0.5647


INFO:62_autogluon_seed_averaging:定着率: 0.5647


[2026-08-15 15:29:12] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:62_autogluon_seed_averaging:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [27]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-15 15:29:12] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:62_autogluon_seed_averaging:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-15 15:29:12] [INFO] Test  早期退職者: 0名 / 2502名


INFO:62_autogluon_seed_averaging:Test  早期退職者: 0名 / 2502名


[2026-08-15 15:29:12] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:62_autogluon_seed_averaging:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-15 15:29:12] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:62_autogluon_seed_averaging:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`51_`と同一ロジック）

In [28]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [29]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-15 15:29:12] [INFO] ------------------------------------------------------------


INFO:62_autogluon_seed_averaging:------------------------------------------------------------


[2026-08-15 15:29:12] [INFO] split非依存の基本特徴量を生成中...


INFO:62_autogluon_seed_averaging:split非依存の基本特徴量を生成中...


[2026-08-15 15:29:12] [INFO] ------------------------------------------------------------


INFO:62_autogluon_seed_averaging:------------------------------------------------------------


[2026-08-15 15:34:53] [INFO] split非依存の基本特徴量生成完了


INFO:62_autogluon_seed_averaging:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`51_`と同一・継続採用）

In [30]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-15 15:34:53] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:62_autogluon_seed_averaging:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-15 15:34:54] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:62_autogluon_seed_averaging:入社時メモ: SVD累積寄与率=0.760


[2026-08-15 15:34:58] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:62_autogluon_seed_averaging:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-15 15:34:59] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:62_autogluon_seed_averaging:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-15 15:34:59] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:62_autogluon_seed_averaging:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`51_`と同一・継続採用）

In [31]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-15 15:34:59] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:62_autogluon_seed_averaging:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-15 15:37:03] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:62_autogluon_seed_averaging:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`51_`と同一）

In [32]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-15 15:37:03] [INFO] Persona単位の基本特徴量を生成中...


INFO:62_autogluon_seed_averaging:Persona単位の基本特徴量を生成中...


[2026-08-15 15:37:03] [INFO] Persona単位の基本特徴量処理完了


INFO:62_autogluon_seed_averaging:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版、`51_`と同一）

`51_`と同じくv1/v2両方を生成するが、実際に特徴量として使うのはv2（`BLOCK={"L2"}`）のみ
（`28_`以降ずっとv2が現在の最良）。


In [33]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 50_: 見出しがない書式B（276件、5.24%）のフォールバック（49_で確認済み・Public -0.0022〜-0.0035）。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-15 15:37:03] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:62_autogluon_seed_averaging:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-15 15:37:04] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:62_autogluon_seed_averaging:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-15 15:37:04] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:62_autogluon_seed_averaging:L_v2: Train (2761, 3), Test (2502, 3)


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数（`51_`と同一）

In [34]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（51_と完全に同一ロジック）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")


✅ 部署Target Encoding・prepare_split関数定義完了


## 7. 特徴量の組み立て（`51_`と同一、`BLOCK={"L2"}`固定）

In [35]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-15 15:37:04] [INFO] ============================================================


INFO:62_autogluon_seed_averaging:============================================================


[2026-08-15 15:37:04] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:62_autogluon_seed_averaging:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-15 15:37:04] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:62_autogluon_seed_averaging:main_train=2208, main_valid(生存者)=535


[2026-08-15 15:37:04] [INFO] 全件=2761


INFO:62_autogluon_seed_averaging:全件=2761


[2026-08-15 15:37:04] [INFO] 特徴量数: 441


INFO:62_autogluon_seed_averaging:特徴量数: 441


## 8. 特徴量グループの棚卸し（`51_`から移植、内容は同一）

In [36]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


## 9. AutoGluon の設定（`50_`＝現最良の生成元と同一。変えるのはシードと`num_bag_folds`のみ）

In [37]:
import pandas as pd
pd.set_option("mode.chained_assignment", None)

FEATURE_SETS = {"full441": {"groups": ALL_GROUPS}}


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


for _n, _s in FEATURE_SETS.items():
    _c = cols_for(_s, ag_full)
    assert _c == cols_for(_s, ag_train_80b), f"{_n}: 80%学習と全件で列が食い違う"
    assert len(_c) == 441, f"{_n}: {len(_c)}列（441列のはず）"
    print(f"  {_n}: {len(_c)} 列 ✅")

PRESETS = "best_quality"
TIME_LIMIT = 7200            # 50_/51_と同一。時間予算の拡大は61_で別途検証中
AG_METRIC = "log_loss"
EXCLUDED_MODELS = ["FASTAI", "NN_TORCH", "KNN"]
DYNAMIC_STACKING = False
NUM_STACK_LEVELS = 1

# 62_ の変数はここだけ
AG_SEEDS = [42, 2024, 7]     # 独立実行して平均するシード
NUM_BAG_FOLDS_BASE = 8       # 50_/51_と同一
NUM_BAG_FOLDS_ALT = 16       # 追加検証: foldを倍にする

NOISE_FLOOR_P95 = 0.02122

print(f"\npresets={PRESETS} / time_limit={TIME_LIMIT}秒/fit / eval_metric={AG_METRIC}")
print(f"除外モデル: {EXCLUDED_MODELS}")
print(f"シード平均: {AG_SEEDS}（num_bag_folds={NUM_BAG_FOLDS_BASE}）")
print(f"追加構成  : num_bag_folds={NUM_BAG_FOLDS_ALT}（シード{AG_SEEDS[0]}のみ）")
print(f"想定所要時間: full run {len(AG_SEEDS)+1}本 × {TIME_LIMIT/3600:.0f}時間 = 約{(len(AG_SEEDS)+1)*TIME_LIMIT/3600:.0f}時間")


  full441: 441 列 ✅

presets=best_quality / time_limit=7200秒/fit / eval_metric=log_loss
除外モデル: ['FASTAI', 'NN_TORCH', 'KNN']
シード平均: [42, 2024, 7]（num_bag_folds=8）
追加構成  : num_bag_folds=16（シード42のみ）
想定所要時間: full run 4本 × 2時間 = 約8時間


## 10. AutoGluon の学習関数（`51_`と同一 + シード制御）

In [38]:
!pip install -q autogluon.tabular
# ⚠️ ray は入れない。47_ では ray 2.56.1 の pyarrow が Colab のものと
#    バイナリ非互換になり、ParallelLocalFoldFittingStrategy 経由で fold を切る
#    全モデル（CatBoost/LightGBM/XGBoost/NN）が学習前に落ちた。


In [39]:
import random
from autogluon.tabular import TabularPredictor

try:
    import ray  # noqa: F401
    print("⚠️ ray が入っている。47_ ではこれが原因で全モデルが落ちた。ランタイムを初期化すること。")
except ImportError:
    print("✅ ray は入っていない（正常）。foldは逐次学習される。")

AG_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
AG_DIR.mkdir(parents=True, exist_ok=True)


def _frame(df, feats, with_label=True):
    """AutoGluon に渡すフレーム。明示的に.copy()してSettingWithCopyErrorを避ける（44_の教訓）。"""
    cols = list(feats) + ([TARGET_COL] if with_label and TARGET_COL in df.columns else [])
    out = df[cols].copy()
    out.reset_index(drop=True, inplace=True)
    out._is_copy = None
    return out


def fit_autogluon(tag, train_df, feats, seed, n_bag_folds, tuning_df=None, time_limit=TIME_LIMIT):
    """AutoGluon を1回 fit する。保存済みなら読み込むだけ（中断・再接続に備える）。

    62_: シードを変えて独立実行するため、
      (1) numpy/random のグローバルシードを設定
      (2) 学習データの行順をシードでシャッフル（bagging の fold 割り当てが変わる）
    の2点でランダム性を制御する。

    注意: AutoGluonの `fit()` は `random_seed` 引数を受け付けない（有効kwargs一覧に無く、
    渡すと ValueError になる）。また各モデルのシードを変えるには `hyperparameters` を
    上書きする必要があるが、それは zeroshot ポートフォリオを壊すので行わない（47_の教訓）。
    したがって実行間の差は**fold割り当ての違いのみ**から生じる。
    → 第12節で測るシード間MADは、真の実行間変動の**下限**として解釈すること。
    """
    path = AG_DIR / tag
    if (path / "predictor.pkl").exists():
        logger.info(f"[{tag}] 保存済みpredictorを読み込む（再学習しない）")
        return TabularPredictor.load(str(path))

    np.random.seed(seed)
    random.seed(seed)
    tr = _frame(train_df, feats).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    logger.info("=" * 60)
    logger.info(f"[{tag}] AutoGluon fit: n={len(tr)}, 特徴量={len(feats)}, "
                f"seed={seed}, num_bag_folds={n_bag_folds}, time_limit={time_limit}")
    kw = dict(presets=PRESETS, time_limit=time_limit,
              excluded_model_types=EXCLUDED_MODELS,
              num_bag_folds=n_bag_folds, num_stack_levels=NUM_STACK_LEVELS)
    if tuning_df is not None:
        kw["tuning_data"] = _frame(tuning_df, feats)
        kw["use_bag_holdout"] = True
    else:
        kw["dynamic_stacking"] = DYNAMIC_STACKING
    p = TabularPredictor(label=TARGET_COL, eval_metric=AG_METRIC, path=str(path),
                         problem_type="binary")
    # AutoGluon の fit は random_seed を受け取らない（ValueError になる）。
    # ランダム性の制御は上のグローバルシード + 行シャッフルのみで行う。
    p.fit(tr, **kw)
    return p


def leaderboard(predictor, data=None):
    try:
        return predictor.leaderboard(data, silent=True) if data is not None \
            else predictor.leaderboard(silent=True)
    except TypeError:
        return predictor.leaderboard(data) if data is not None else predictor.leaderboard()


def positive_proba_model(predictor, X, model=None):
    pp = predictor.predict_proba(X, model=model)
    pos = predictor.positive_class if hasattr(predictor, "positive_class") else None
    if pos is None or pos not in pp.columns:
        pos = 1 if 1 in pp.columns else pp.columns[-1]
    return pp[pos].values


def pick(lb, kind):
    w = lb["model"].str.startswith("WeightedEnsemble")
    names = lb[w]["model"].tolist() if kind == "weighted" else lb[~w]["model"].tolist()
    return names[0] if names else None


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ AutoGluon 関数定義完了（シード制御付き）")


✅ ray は入っていない（正常）。foldは逐次学習される。
✅ AutoGluon 関数定義完了（シード制御付き）


## 11. 実行: 3シードの独立実行 + bag16

各シードで Train全件 fit → Test予測（weighted / best_single 両方を保存）。
検証用の holdout run は行わない —— [[validation-asymmetry]]の通り検証は採否の根拠にできず、
今回は「同一構成の分散削減」なので検証で選別する対象が無いため、時間予算を full run に全振りする。


In [40]:
feats = cols_for(FEATURE_SETS["full441"], ag_full)
Xte = _frame(test_features_full, feats, with_label=False)

runs = {}          # tag -> {kind: preds}
for seed in AG_SEEDS:
    tag = f"full441_seed{seed}"
    p = fit_autogluon(tag, ag_full, feats, seed=seed, n_bag_folds=NUM_BAG_FOLDS_BASE)
    lb = leaderboard(p)
    lb.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_leaderboard_{tag}.csv", index=False)
    print(f"\n===== [{tag}] 完走モデル {len(lb)}件 =====")
    print(lb[["model", "score_val", "fit_time"]].head(8).to_string(index=False))
    runs[tag] = {}
    for kind in ["weighted", "best_single"]:
        mdl = pick(lb, kind)
        if mdl is None:
            print(f"  {kind}: 該当モデルなし"); continue
        pr = positive_proba_model(p, Xte, mdl)
        runs[tag][kind] = pr
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{tag}_{kind}_testpreds.npy", pr)
        logger.info(f"  [{tag}] {kind}: model={mdl}, 予測平均={pr.mean():.4f}")

tag16 = f"full441_bag16_seed{AG_SEEDS[0]}"
p16 = fit_autogluon(tag16, ag_full, feats, seed=AG_SEEDS[0], n_bag_folds=NUM_BAG_FOLDS_ALT)
lb16 = leaderboard(p16)
lb16.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_leaderboard_{tag16}.csv", index=False)
print(f"\n===== [{tag16}] 完走モデル {len(lb16)}件 =====")
print(lb16[["model", "score_val", "fit_time"]].head(8).to_string(index=False))
runs[tag16] = {}
for kind in ["weighted", "best_single"]:
    mdl = pick(lb16, kind)
    if mdl is None: continue
    pr = positive_proba_model(p16, Xte, mdl)
    runs[tag16][kind] = pr
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{tag16}_{kind}_testpreds.npy", pr)

logger.info("全run完了")


[2026-08-15 15:37:08] [INFO] ============================================================


INFO:62_autogluon_seed_averaging:============================================================


[2026-08-15 15:37:08] [INFO] [full441_seed42] AutoGluon fit: n=2761, 特徴量=441, seed=42, num_bag_folds=8, time_limit=7200


INFO:62_autogluon_seed_averaging:[full441_seed42] AutoGluon fit: n=2761, 特徴量=441, seed=42, num_bag_folds=8, time_limit=7200
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          8
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       49.07 GB / 50.99 GB (96.2%)
Disk Space Avail:   194.51 GB / 225.83 GB (86.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 7200s
AutoGluon will save models to "/content/drive/MyDrive/jaggle_2026/saved_models/20260815/62_autogluon_seed_averaging/full441_seed42"
Train Data Rows:    2761
Train Data Columns: 441
Label Column:      

[1000]	valid_set's binary_logloss: 0.503461
[1000]	valid_set's binary_logloss: 0.494114


...

	-0.5122	 = Validation score   (-log_loss)
	42.58s	 = Training   runtime
	0.1s	 = Validation runtime
Fitting model: XGBoost_r33_BAG_L1 ... Training model for up to 2920.69s of the 5321.75s of remaining time.
	Fitting 16 child models (S1F1 - S1F16) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=4, gpus=0)
	-0.5238	 = Validation score   (-log_loss)
	297.79s	 = Training   runtime
	0.14s	 = Validation runtime
Fitting model: ExtraTrees_r42_BAG_L1 ... Training model for up to 2622.07s of the 5023.13s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=8, gpus=0, mem=0.0/47.9 GB
	-0.5482	 = Validation score   (-log_loss)
	2.88s	 = Training   runtime
	0.21s	 = Validation runtime
Fitting model: CatBoost_r137_BAG_L1 ... Training model for up to 2618.83s of the 5019.89s of remaining time.
	Fitting 16 child models (S1F1 - S1F16) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=4, gpus=0)
	-0.4926	 = Validation score   (-l

[1000]	valid_set's binary_logloss: 0.476691
[1000]	valid_set's binary_logloss: 0.475307


	-0.5192	 = Validation score   (-log_loss)
	198.02s	 = Training   runtime
	0.14s	 = Validation runtime
Fitting model: RandomForest_r39_BAG_L1 ... Training model for up to 391.19s of the 2792.25s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=8, gpus=0, mem=0.0/48.3 GB
	-0.5627	 = Validation score   (-log_loss)
	27.12s	 = Training   runtime
	0.2s	 = Validation runtime
Fitting model: CatBoost_r167_BAG_L1 ... Training model for up to 363.72s of the 2764.78s of remaining time.
	Fitting 16 child models (S1F1 - S1F16) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=4, gpus=0)
	-0.5013	 = Validation score   (-log_loss)
	205.16s	 = Training   runtime
	0.47s	 = Validation runtime
Fitting model: XGBoost_r98_BAG_L1 ... Training model for up to 157.65s of the 2558.71s of remaining time.
	Fitting 16 child models (S1F1 - S1F16) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=4, gpus=0)
	-0.5244	 = Validation score   (-


===== [full441_bag16_seed42] 完走モデル 54件 =====
               model  score_val    fit_time
 WeightedEnsemble_L3  -0.482714 1574.392674
  CatBoost_r9_BAG_L2  -0.484853 1448.053635
     CatBoost_BAG_L2  -0.485445  769.361562
CatBoost_r137_BAG_L2  -0.485694  705.858838
 CatBoost_r13_BAG_L2  -0.486430  998.641470
CatBoost_r177_BAG_L2  -0.487945  735.919646
 WeightedEnsemble_L2  -0.489260  407.783985
   LightGBMXT_BAG_L2  -0.491392  685.562047
[2026-08-15 23:37:32] [INFO] 全run完了


INFO:62_autogluon_seed_averaging:全run完了


## 12. シード間MADの実測（本ノートブックのもう一つの目的）

同一構成・シードだけ違う実行の間で、Test予測がどれだけ動くかを実測する。
[[refit-chaos-noise-floor]]の0.02122は単層CatBoostで測った値で、**AutoGluonでは未測定**。
ここが0.02122に近ければ、これまでの構成比較の大半はノイズを見ていたことになる。

**重要な限定**: AutoGluonは`fit(random_seed=)`に非対応で、モデル個別のシードを変えるには`hyperparameters`の上書きが必要（47_の教訓によりやらない）。よってここでの実行間の差は**fold割り当ての違いのみ**に由来し、測定されるMADは真の実行間変動の**下限**である。下限であっても0.02122に近ければ、結論（構成比較がノイズに埋もれている）はより強く成立する。


In [41]:
seed_tags = [f"full441_seed{s}" for s in AG_SEEDS]
print("=== シード間の予測差（weighted）===")
mads = []
for i in range(len(seed_tags)):
    for j in range(i + 1, len(seed_tags)):
        a, b = runs[seed_tags[i]].get("weighted"), runs[seed_tags[j]].get("weighted")
        if a is None or b is None: continue
        mad = float(np.abs(a - b).mean()); corr = float(np.corrcoef(a, b)[0, 1])
        mads.append(mad)
        print(f"  seed{AG_SEEDS[i]} vs seed{AG_SEEDS[j]}: MAD={mad:.5f}  相関={corr:.5f}")
if mads:
    m = float(np.mean(mads))
    print(f"\n  シード間MAD平均 = {m:.5f}   （単層CatBoostのノイズ床 {NOISE_FLOOR_P95}）")
    if m > NOISE_FLOOR_P95 * 0.5:
        print("  ⚠️ シードを変えるだけでノイズ床に匹敵する差が出る。")
        print("     → これまでの構成間の微差（0.001〜0.004）の多くはノイズだった可能性が高い。")
    else:
        print("  → AutoGluonのシード間変動は単層CatBoostのノイズ床よりは小さい。")


=== シード間の予測差（weighted）===
  seed42 vs seed2024: MAD=0.01962  相関=0.99481
  seed42 vs seed7: MAD=0.02738  相関=0.99197
  seed2024 vs seed7: MAD=0.02443  相関=0.99319

  シード間MAD平均 = 0.02381   （単層CatBoostのノイズ床 0.02122）
  ⚠️ シードを変えるだけでノイズ床に匹敵する差が出る。
     → これまでの構成間の微差（0.001〜0.004）の多くはノイズだった可能性が高い。


## 13. シード平均の作成と提出候補

In [42]:
RESULT_SCHEMA = ["config", "n_features", "n_runs", "n_bag_folds", "kind",
                 "pred_mean", "mad_vs_best", "corr_vs_best", "submission_path"]


def make_row(**kw):
    unknown = set(kw) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}; row.update(kw); return row


_best = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_50_autogluon_memofix_AG50_full441_weighted.csv"))
assert _best, "現最良（50_ AG50_full441_weighted, Public 0.513108）が見つからない"
BEST_PRED = pd.read_csv(_best[-1], header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]
print(f"比較基準: {_best[-1].name}（Public 0.513108, 予測平均 {BEST_PRED.mean():.4f}）")

results = {}
def register(label, preds, n_runs, folds, kind):
    path = save_submission(test_features_full.index, preds, label)
    al = pd.Series(preds, index=test_features_full.index).loc[BEST_PRED.index]
    results[label] = make_row(
        config=label, n_features=len(feats), n_runs=n_runs, n_bag_folds=folds, kind=kind,
        pred_mean=float(preds.mean()),
        mad_vs_best=float(np.abs(al.values - BEST_PRED.values).mean()),
        corr_vs_best=float(np.corrcoef(al.values, BEST_PRED.values)[0, 1]),
        submission_path=path)

for kind in ["weighted", "best_single"]:
    got = [runs[t][kind] for t in seed_tags if kind in runs[t]]
    if len(got) >= 2:
        register(f"AG62_seedavg{len(got)}_{kind}", np.mean(got, axis=0), len(got), NUM_BAG_FOLDS_BASE, kind)
    if kind in runs.get(tag16, {}):
        register(f"AG62_bag16_{kind}", runs[tag16][kind], 1, NUM_BAG_FOLDS_ALT, kind)

summary = pd.DataFrame(list(results.values()))
summary["提出"] = np.where(summary["mad_vs_best"] > NOISE_FLOOR_P95, "提出する",
                            "見送り（現最良との差がノイズ床以下）")
pd.set_option("display.width", 240)
print()
print(summary.to_string(index=False))
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)
pd.DataFrame(list(results.values())).to_csv(CHECKPOINT_PATH, index=False)
logger.info("サマリを保存")

print()
print("=" * 74)
print("提出優先順位（50_/51_ともWeightedEnsembleがbest_singleを上回っている実績に従う）")
print("=" * 74)
_order = [c for c in summary["config"] if c.endswith("weighted")] + \
         [c for c in summary["config"] if c.endswith("best_single")]
_i = 0
for c in _order:
    r = summary[summary["config"] == c]
    if len(r) == 0: continue
    r = r.iloc[0]; _i += 1
    print(f"{_i}. {c:<30s} {Path(r['submission_path']).name}")
    print(f"     runs={int(r['n_runs'])} folds={int(r['n_bag_folds'])} / "
          f"現最良との相関 {r['corr_vs_best']:.5f} / 平均絶対差 {r['mad_vs_best']:.5f} / {r['提出']}")


比較基準: 20260813_50_autogluon_memofix_AG50_full441_weighted.csv（Public 0.513108, 予測平均 0.5862）
[2026-08-15 23:37:33] [INFO]   提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_seedavg3_weighted.csv（予測平均=0.5837）


INFO:62_autogluon_seed_averaging:  提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_seedavg3_weighted.csv（予測平均=0.5837）


[2026-08-15 23:37:33] [INFO]   提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_bag16_weighted.csv（予測平均=0.5880）


INFO:62_autogluon_seed_averaging:  提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_bag16_weighted.csv（予測平均=0.5880）


[2026-08-15 23:37:33] [INFO]   提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_seedavg3_best_single.csv（予測平均=0.5833）


...

[2026-08-15 23:37:33] [INFO]   提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_bag16_best_single.csv（予測平均=0.5928）


INFO:62_autogluon_seed_averaging:  提出ファイル: 20260815_62_autogluon_seed_averaging_AG62_bag16_best_single.csv（予測平均=0.5928）



                   config  n_features  n_runs  n_bag_folds        kind  pred_mean  mad_vs_best  corr_vs_best                                                                                                            submission_path                 提出
   AG62_seedavg3_weighted         441       3            8    weighted   0.583660     0.020159      0.994500    /content/drive/MyDrive/jaggle_2026/data/output/20260815/20260815_62_autogluon_seed_averaging_AG62_seedavg3_weighted.csv 見送り（現最良との差がノイズ床以下）
      AG62_bag16_weighted         441       1           16    weighted   0.587986     0.022346      0.993358       /content/drive/MyDrive/jaggle_2026/data/output/20260815/20260815_62_autogluon_seed_averaging_AG62_bag16_weighted.csv               提出する
AG62_seedavg3_best_single         441       3            8 best_single   0.583310     0.023778      0.993192 /content/drive/MyDrive/jaggle_2026/data/output/20260815/20260815_62_autogluon_seed_averaging_AG62_seedavg3_best_single.csv               

INFO:62_autogluon_seed_averaging:サマリを保存



提出優先順位（50_/51_ともWeightedEnsembleがbest_singleを上回っている実績に従う）
1. AG62_seedavg3_weighted         20260815_62_autogluon_seed_averaging_AG62_seedavg3_weighted.csv
     runs=3 folds=8 / 現最良との相関 0.99450 / 平均絶対差 0.02016 / 見送り（現最良との差がノイズ床以下）
2. AG62_bag16_weighted            20260815_62_autogluon_seed_averaging_AG62_bag16_weighted.csv
     runs=1 folds=16 / 現最良との相関 0.99336 / 平均絶対差 0.02235 / 提出する
3. AG62_seedavg3_best_single      20260815_62_autogluon_seed_averaging_AG62_seedavg3_best_single.csv
     runs=3 folds=8 / 現最良との相関 0.99319 / 平均絶対差 0.02378 / 提出する
4. AG62_bag16_best_single         20260815_62_autogluon_seed_averaging_AG62_bag16_best_single.csv
     runs=1 folds=16 / 現最良との相関 0.99229 / 平均絶対差 0.02797 / 提出する


## 14. 提出方針

- **採否はPublicのみ**（[[validation-asymmetry]]）。本ノートブックは holdout run を行っていないので
  検証スコアは存在しない。判断材料は「現最良との予測差」と「シード間MAD」だけ。
- **第一候補は `AG62_seedavg3_weighted`**。`50_`/`51_`ともWeightedEnsembleがbest_singleを
  上回っている実績（[[ensemble-oof-overfitting]]の addendum 5）に従う。
- `mad_vs_best` がノイズ床(0.02122)以下なら、シード平均は現最良と実質同じ予測ということなので、
  提出しても情報が得られない。その場合は**第12節のシード間MADの実測値そのものが成果**
  （AutoGluonの分散が小さいと分かれば、これまでの構成比較の信頼性が上がる）。

### この実験で分かること（どちらに転んでも情報になる）

| シード間MAD | 解釈 | 次の一手 |
|---|---|---|
| 大きい（>0.01）| これまでの構成間の微差はノイズだった。個々の構成比較は信用できない | シード平均を標準手順にする |
| 小さい（<0.005）| AutoGluonは安定。構成比較の差は本物 | 構成探索に戻ってよい |
